# Dijkstra's Shortest-Path Algorithm
**EE4033 Algorithms — Lecture 11 (NTU, Spring 2025)**

---

## Goal
Given a directed graph $G=(V,E)$ with **non-negative** edge weights and a source node $s$, find the **minimum-weight path** from $s$ to every other vertex.

## Key Ideas
- Maintain a set $S$ of vertices whose **final** shortest-path distances are already known.
- Repeatedly **greedily** pick the vertex $u \notin S$ with the smallest current distance estimate $u.d$, add it to $S$, then **relax** all edges leaving $u$.
- **Relax(u, v, w):** if $v.d > u.d + w(u,v)$, update $v.d \leftarrow u.d + w(u,v)$ and $v.\pi \leftarrow u$.

## Pseudocode
```
Dijkstra(G, w, s):
  Initialize-Single-Source(G, s)   # v.d = ∞ for all v; s.d = 0
  S = ∅
  Q = G.V                          # priority queue (min-heap on .d)
  while Q ≠ ∅:
      u = Extract-Min(Q)
      S = S ∪ {u}
      for each v ∈ G.Adj[u]:
          Relax(u, v, w)
```

## Time Complexity
| Priority Queue | Extract-Min | Decrease-Key | Total |
|---|---|---|---|
| Linear array | $O(V)$ | $O(1)$ | $O(V^2)$ |
| Binary heap  | $O(\log V)$ | $O(\log V)$ | $O(E \log V)$ |
| Fibonacci heap | $O(\log V)$ amortized | $O(1)$ amortized | $O(E + V \log V)$ |


---
## Setup — Install & Import

In [ ]:
# Install visualization libraries (already present in most Colab environments)
!pip install -q matplotlib networkx ipywidgets

In [ ]:
import heapq
import math
import networkx as nx
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import ipywidgets as widgets
from IPython.display import display, clear_output
from copy import deepcopy

---
## Part 1 — Core Implementation

Below is a clean Python implementation that closely mirrors the pseudocode from lecture.

In [ ]:
def dijkstra(graph, source):
    """
    graph: dict  {u: [(v, weight), ...]}  — adjacency list
    source: starting vertex
    Returns: (dist, prev)
      dist[v]  = shortest distance from source to v
      prev[v]  = predecessor of v on the shortest path (v.π)
    """
    # --- Initialize-Single-Source ---
    dist = {v: math.inf for v in graph}   # v.d = ∞
    prev = {v: None     for v in graph}   # v.π = NIL
    dist[source] = 0                       # s.d = 0

    # Priority queue entries: (distance_estimate, vertex)
    # Python's heapq is a min-heap, matching Extract-Min(Q)
    pq = [(0, source)]

    explored = set()   # S in the pseudocode

    while pq:
        d_u, u = heapq.heappop(pq)   # Extract-Min(Q)

        if u in explored:             # stale entry — skip
            continue
        explored.add(u)               # S = S ∪ {u}

        for v, w in graph[u]:         # for each v ∈ G.Adj[u]
            # --- Relax(u, v, w) ---
            if dist[u] + w < dist[v]:
                dist[v] = dist[u] + w
                prev[v] = u
                heapq.heappush(pq, (dist[v], v))  # update priority

    return dist, prev


def reconstruct_path(prev, source, target):
    """Trace predecessor pointers back from target to source."""
    path = []
    node = target
    while node is not None:
        path.append(node)
        node = prev[node]
    path.reverse()
    return path if path[0] == source else []  # empty if unreachable

### Run on the Lecture Example

The graph from slide 14: vertices $\{s, t, x, y, z\}$ with directed edges.

In [ ]:
# Directed graph from the lecture example (slide 14)
lecture_graph = {
    's': [('t', 10), ('y', 5)],
    't': [('x', 1),  ('y', 2)],
    'x': [('z', 4)],
    'y': [('t', 3),  ('x', 9), ('z', 2)],
    'z': [('x', 6), ('s', 7)],
}

dist, prev = dijkstra(lecture_graph, 's')

print("Shortest distances from 's':")
for v in sorted(dist):
    path = reconstruct_path(prev, 's', v)
    print(f"  s -> {v}: distance = {dist[v]:>3}  |  path = {' -> '.join(path)}")

Expected output (from lecture slide 14):
```
s -> s:  0   (trivially)
s -> t:  8   (s -> y -> t)
s -> x:  9   (s -> y -> t -> x)
s -> y:  5   (s -> y)
s -> z:  7   (s -> y -> z)
```

---
## Part 2 — Step-by-Step Visualization

This cell records every state of the algorithm so we can replay it.

In [ ]:
def dijkstra_steps(graph, source):
    """Same algorithm but records a snapshot after each Extract-Min."""
    dist = {v: math.inf for v in graph}
    prev = {v: None     for v in graph}
    dist[source] = 0
    pq = [(0, source)]
    explored = set()
    steps = []  # list of snapshots

    while pq:
        d_u, u = heapq.heappop(pq)
        if u in explored:
            continue
        explored.add(u)

        relaxed_edges = []
        for v, w in graph[u]:
            if dist[u] + w < dist[v]:
                dist[v] = dist[u] + w
                prev[v] = u
                heapq.heappush(pq, (dist[v], v))
                relaxed_edges.append((u, v))

        # Snapshot: copy current state
        steps.append({
            'current': u,
            'explored': set(explored),
            'dist': dict(dist),
            'prev': dict(prev),
            'relaxed': relaxed_edges,
        })

    return steps


# Fixed node positions matching the lecture diagram layout
POS = {
    's': (0, 1),
    't': (2, 2),
    'x': (4, 2),
    'y': (2, 0),
    'z': (4, 0),
}

def draw_step(G_nx, step, step_num, total):
    """Draw a single algorithm snapshot on a matplotlib figure."""
    explored = step['explored']
    current  = step['current']
    dist_map = step['dist']
    prev_map = step['prev']
    relaxed  = step['relaxed']

    # Node colors: black=explored, gray=current (just extracted), white=queue
    node_colors = []
    for n in G_nx.nodes():
        if n == current:
            node_colors.append('#aaaaaa')  # current (being processed)
        elif n in explored:
            node_colors.append('#333333')  # fully explored
        else:
            node_colors.append('white')    # still in Q

    # Edge colors: pink=predecessor tree, red=just relaxed, gray=others
    pred_edges = {(prev_map[v], v) for v in prev_map if prev_map[v] is not None}
    edge_colors = []
    for u, v in G_nx.edges():
        if (u, v) in relaxed:
            edge_colors.append('red')
        elif (u, v) in pred_edges:
            edge_colors.append('#e87070')
        else:
            edge_colors.append('#cccccc')

    fig, ax = plt.subplots(figsize=(8, 5))
    ax.set_title(f"Step {step_num}/{total} — Extract-Min → '{current}'   "
                 f"(explored: {sorted(explored)})", fontsize=12)

    # Draw graph
    nx.draw_networkx(G_nx, POS, ax=ax,
                     node_color=node_colors, node_size=900,
                     edge_color=edge_colors, width=2.5,
                     font_color='white', font_weight='bold',
                     arrows=True, arrowsize=20,
                     connectionstyle='arc3,rad=0.08')

    # Draw edge weight labels
    edge_labels = nx.get_edge_attributes(G_nx, 'weight')
    nx.draw_networkx_edge_labels(G_nx, POS, edge_labels=edge_labels,
                                  ax=ax, font_size=9)

    # Draw distance estimates below each node
    for node, (x, y) in POS.items():
        d = dist_map[node]
        label = str(d) if d != math.inf else '∞'
        ax.text(x, y - 0.28, f'd={label}', ha='center', fontsize=9,
                color='navy', fontweight='bold')

    # Legend
    legend_handles = [
        mpatches.Patch(color='#333333', label='In S (explored)'),
        mpatches.Patch(color='#aaaaaa', label='u (just extracted)'),
        mpatches.Patch(color='white',   label='In Q (unvisited)', ec='black'),
        mpatches.Patch(color='red',     label='Relaxed this step'),
        mpatches.Patch(color='#e87070', label='Predecessor edge'),
    ]
    ax.legend(handles=legend_handles, loc='upper right', fontsize=8)
    ax.axis('off')
    plt.tight_layout()
    plt.show()

In [ ]:
# Build a NetworkX DiGraph for visualization
G_nx = nx.DiGraph()
for u, neighbors in lecture_graph.items():
    for v, w in neighbors:
        G_nx.add_edge(u, v, weight=w)

# Record all algorithm steps
steps = dijkstra_steps(lecture_graph, 's')
print(f"Algorithm completes in {len(steps)} Extract-Min operations.")

---
## Part 3 — Interactive Step-Through Widget

Use the slider (or buttons) to walk through each iteration of the algorithm.

In [ ]:
step_slider = widgets.IntSlider(
    value=1, min=1, max=len(steps),
    description='Step:', continuous_update=False,
    style={'description_width': 'initial'}, layout=widgets.Layout(width='60%')
)

btn_prev = widgets.Button(description='◀ Prev', button_style='info')
btn_next = widgets.Button(description='Next ▶', button_style='info')

out = widgets.Output()

def render(step_idx):
    with out:
        clear_output(wait=True)
        draw_step(G_nx, steps[step_idx - 1], step_idx, len(steps))

def on_slider_change(change):
    render(change['new'])

def on_prev(_):
    if step_slider.value > 1:
        step_slider.value -= 1

def on_next(_):
    if step_slider.value < len(steps):
        step_slider.value += 1

step_slider.observe(on_slider_change, names='value')
btn_prev.on_click(on_prev)
btn_next.on_click(on_next)

controls = widgets.HBox([btn_prev, step_slider, btn_next])
display(controls, out)
render(1)  # show step 1 initially

---
## Part 4 — The Relax Operation in Detail

In [ ]:
def relax(dist, prev, u, v, w):
    """Relax edge (u, v) with weight w. Returns True if distance improved."""
    if dist[u] + w < dist[v]:
        print(f"  Relax({u},{v}): {dist[v]} > {dist[u]} + {w} = {dist[u]+w}  → UPDATE d[{v}] = {dist[u]+w}")
        dist[v] = dist[u] + w
        prev[v] = u
        return True
    else:
        print(f"  Relax({u},{v}): {dist[v]} ≤ {dist[u]} + {w} = {dist[u]+w}  → no change")
        return False

# Trace through the first two iterations manually
print("=== Iteration 1: Extract-Min → s (d=0) ===")
d = {'s': 0, 't': math.inf, 'x': math.inf, 'y': math.inf, 'z': math.inf}
p = {v: None for v in d}
relax(d, p, 's', 't', 10)
relax(d, p, 's', 'y', 5)

print("\n=== Iteration 2: Extract-Min → y (d=5) ===")
relax(d, p, 'y', 't', 3)
relax(d, p, 'y', 'x', 9)
relax(d, p, 'y', 'z', 2)

print("\nDistance estimates after 2 iterations:")
for v in ['s','t','x','y','z']:
    print(f"  d[{v}] = {d[v]}")

---
## Part 5 — Why Negative Weights Break Dijkstra

Dijkstra's correctness relies on **non-negative edge weights**. Let's see what happens when we add a negative edge.

In [ ]:
# Graph where Dijkstra gives the WRONG answer due to a negative edge
negative_graph = {
    'A': [('B', 4), ('C', 1)],
    'B': [('D', 1)],
    'C': [('B', -3)],  # negative edge!
    'D': [],
}

dist_dijkstra, _ = dijkstra(negative_graph, 'A')
print("Dijkstra's result (WRONG due to negative edge):")
for v in ['A','B','C','D']:
    print(f"  A -> {v}: {dist_dijkstra[v]}")

# True answer: A->C->B->D = 1 + (-3) + 1 = -1, but Dijkstra says A->B->D = 5
print("\nTrue shortest distances:")
print("  A -> A: 0")
print("  A -> B: -2  (A->C->B = 1 + (-3) = -2)")
print("  A -> C: 1")
print("  A -> D: -1  (A->C->B->D = -2 + 1 = -1)")
print("\n→ Use Bellman-Ford for graphs with negative edges.")

---
## Part 6 — Try Your Own Graph (Interactive)

Edit the graph below and choose a source node to run Dijkstra interactively.

In [ ]:
# --- Edit this graph ---
# Format: 'node': [('neighbor', weight), ...]
custom_graph = {
    'A': [('B', 7), ('C', 9), ('F', 14)],
    'B': [('C', 10), ('D', 15)],
    'C': [('D', 11), ('F', 2)],
    'D': [('E', 6)],
    'E': [('F', 9)],
    'F': [('E', 9)],
}
custom_source = 'A'  # change source here
# ----------------------

dist_c, prev_c = dijkstra(custom_graph, custom_source)

print(f"Shortest distances from '{custom_source}':")
for v in sorted(dist_c):
    path = reconstruct_path(prev_c, custom_source, v)
    d_str = str(dist_c[v]) if dist_c[v] != math.inf else '∞ (unreachable)'
    p_str = ' -> '.join(path) if path else 'N/A'
    print(f"  {custom_source} -> {v}: distance = {d_str:<6}  path = {p_str}")

# Visualize the result
G_custom = nx.DiGraph()
for u, neighbors in custom_graph.items():
    for v, w in neighbors:
        G_custom.add_edge(u, v, weight=w)

pos_custom = nx.spring_layout(G_custom, seed=42)

# Color shortest-path tree edges
tree_edges = {(prev_c[v], v) for v in prev_c if prev_c[v] is not None}
edge_colors = ['#e87070' if (u,v) in tree_edges else '#cccccc'
               for u, v in G_custom.edges()]

fig, ax = plt.subplots(figsize=(8, 5))
ax.set_title(f"Dijkstra result — source='{custom_source}'  (pink = shortest-path tree)",
             fontsize=11)
nx.draw_networkx(G_custom, pos_custom, ax=ax, node_size=800,
                 node_color='#4a90d9', font_color='white', font_weight='bold',
                 edge_color=edge_colors, width=2.5, arrows=True, arrowsize=18,
                 connectionstyle='arc3,rad=0.08')
nx.draw_networkx_edge_labels(G_custom, pos_custom,
                              edge_labels=nx.get_edge_attributes(G_custom, 'weight'),
                              ax=ax, font_size=9)
for node, (x, y) in pos_custom.items():
    d = dist_c[node]
    ax.text(x, y - 0.12, f"d={d}", ha='center', fontsize=9,
            color='navy', fontweight='bold')
ax.axis('off')
plt.tight_layout()
plt.show()

---
## Part 7 — Time Complexity Demo

Compare runtime growth of the $O(V^2)$ (array-based) vs $O(E \log V)$ (heap-based) implementations on random graphs.

In [ ]:
import time
import random

def dijkstra_array(graph, source):
    """O(V^2) implementation using a plain dict as the priority queue."""
    dist = {v: math.inf for v in graph}
    dist[source] = 0
    unvisited = set(graph.keys())

    while unvisited:
        # O(V) linear scan — the bottleneck
        u = min(unvisited, key=lambda v: dist[v])
        if dist[u] == math.inf:
            break
        unvisited.remove(u)
        for v, w in graph[u]:
            if dist[u] + w < dist[v]:
                dist[v] = dist[u] + w
    return dist


def random_graph(n_nodes, density=3, max_weight=20, seed=0):
    """Generate a random directed graph with ~density edges per node."""
    rng = random.Random(seed)
    nodes = list(range(n_nodes))
    graph = {v: [] for v in nodes}
    for u in nodes:
        neighbors = rng.sample([v for v in nodes if v != u],
                                k=min(density, n_nodes - 1))
        for v in neighbors:
            graph[u].append((v, rng.randint(1, max_weight)))
    return graph


sizes = [50, 100, 200, 400, 800]
t_heap  = []
t_array = []

for n in sizes:
    g = random_graph(n)

    t0 = time.perf_counter()
    for _ in range(5): dijkstra(g, 0)          # heap-based
    t_heap.append((time.perf_counter() - t0) / 5)

    t0 = time.perf_counter()
    for _ in range(5): dijkstra_array(g, 0)    # array-based
    t_array.append((time.perf_counter() - t0) / 5)

fig, ax = plt.subplots(figsize=(7, 4))
ax.plot(sizes, t_heap,  'o-', label='Heap  O(E log V)', color='steelblue')
ax.plot(sizes, t_array, 's-', label='Array O(V²)',      color='tomato')
ax.set_xlabel('Number of Vertices (V)')
ax.set_ylabel('Time (seconds)')
ax.set_title('Heap-based vs Array-based Dijkstra Runtime')
ax.legend()
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

---
## Part 8 — Student Exercises

### Exercise 1 — Trace the Practice Problem from Lecture
The practice graph (slide 15) has vertices $\{s, a, b, c, d, e\}$. Verify your hand-traced answers by running Dijkstra below.

In [ ]:
# Practice graph from slide 15
# TODO: Fill in the edges from the lecture diagram and verify your hand-trace
practice_graph = {
    's': [('a', 10), ('b', 5)],
    'a': [('c', 1),  ('b', 2)],
    'b': [('a', 3),  ('c', 9), ('d', 2)],
    'c': [('e', 4)],
    'd': [('e', 6),  ('c', 7)],  # fill in remaining edges
    'e': [],
}

# Run Dijkstra
dist_p, prev_p = dijkstra(practice_graph, 's')

print("Distances from 's' in practice graph:")
for v in sorted(dist_p):
    path = reconstruct_path(prev_p, 's', v)
    print(f"  s -> {v}: {dist_p[v]}   path: {' -> '.join(path)}")

# Does d[a]=8, d[b]=5, d[c]=9, d[d]=7, d[e]=13 match the lecture answer?

### Exercise 2 — Implement Dijkstra from Scratch

Fill in the missing lines to complete Dijkstra using a binary min-heap.

In [ ]:
import heapq, math

def dijkstra_student(graph, source):
    dist = {v: math.inf for v in graph}
    prev = {v: None     for v in graph}
    dist[source] = 0

    pq = [(0, source)]
    visited = set()

    while pq:
        d_u, u = heapq.heappop(pq)

        if u in visited:
            continue
        visited.add(u)

        for v, w in graph[u]:
            # TODO: implement the Relax step here
            # if ... < dist[v]:
            #     dist[v] = ...
            #     prev[v] = ...
            #     heapq.heappush(...)
            pass  # remove this line when you fill in the code

    return dist, prev

# Test your implementation:
dist_s, _ = dijkstra_student(lecture_graph, 's')
expected  = {'s': 0, 't': 8, 'x': 9, 'y': 5, 'z': 7}
if dist_s == expected:
    print("Correct! Your implementation matches the expected output.")
else:
    print(f"Not quite. Got: {dist_s}")
    print(f"Expected:       {expected}")

### Exercise 3 — Conceptual Questions

Answer the following questions in the cell below:

1. **Why must edge weights be non-negative for Dijkstra to work correctly?**  
   *(Hint: think about the loop invariant — once a node is in S, its distance is final.)*

2. **How does Dijkstra differ from Prim's MST algorithm?**  
   *(Hint: what quantity does each algorithm minimize when choosing the next vertex?)*

3. **When would you prefer the $O(V^2)$ (array) implementation over the $O(E \log V)$ (heap) implementation?**  
   *(Hint: think about dense graphs where $E \approx V^2$.)*

4. **Trace the algorithm on this graph and find d[D]:**
   ```
   A --(1)--> B --(2)--> D
   A --(6)--> C --(1)--> D
   B --(3)--> C
   ```
   Source = A. What is d[D] and the shortest path?


In [ ]:
# Exercise 3.4 — verify your answer here
q4_graph = {
    'A': [('B', 1), ('C', 6)],
    'B': [('D', 2), ('C', 3)],
    'C': [('D', 1)],
    'D': [],
}

d4, p4 = dijkstra(q4_graph, 'A')
path_D = reconstruct_path(p4, 'A', 'D')
print(f"d[D] = {d4['D']}   path: {' -> '.join(path_D)}")
# Does this match your hand-traced answer?

---
## Summary

| Concept | Key Point |
|---|---|
| **Goal** | Single-source shortest paths on non-negative weighted directed graphs |
| **Greedy choice** | Always expand the unvisited vertex with the smallest current estimate |
| **Relax** | Update $v.d \leftarrow u.d + w(u,v)$ if it improves the estimate |
| **Correctness** | Loop invariant: once in $S$, $v.d = \delta(s,v)$ (proven by induction on $|S|$) |
| **Complexity** | $O(V^2)$ array · $O(E\log V)$ heap · $O(E + V\log V)$ Fibonacci heap |
| **Limitation** | Fails with negative edge weights — use Bellman-Ford instead |
